## 모듈 불러오기

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import platform
warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# 음수 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 출력 짤림 방지
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


print("=" * 60)
print("로드 완료")
print("=" * 60)

로드 완료


## 데이터 불러오기

In [3]:
# 원본 데이터 불러오기
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv")

In [15]:
# 전처리용 DataFrame 생성
pre = df.copy()

# 1차 전처리
- 파생변수 생성
- 결측치 처리
- 미사용 컬럼 제거
- 행 제거


In [ ]:
# 파생변수 생성

# 파생변수 1.종속변수(target) 생성
status_map = { 
    "Fully Paid" : 0 ,
    "Charged Off" :1, 
    "Default" : 1
}
pre["target"] = pre["loan_status"].map(status_map)
pre = pre[pre["target"].notnull()]


# 파생변수 2. 날짜형 변수 년, 월 단위 변수(issue_year, issue_month) 생성 
pre['issue_d'] = pd.to_datetime(pre['issue_d'], errors="coerce") # 변수(issue_d)를 날짜형으로 변환
pre['issue_year'] = pre['issue_d'].dt.year.astype(str) # str 타입으로 저장 --> 모델링에 사용할 때 문자형으로 입력 예정
pre['issue_month'] = pre['issue_d'].dt.month.astype(str) # str 타입으로 저장 --> 모델링에 사용할 때 문자형으로 입력 예정

# 파생변수 3. 월 상환 부담율(installment_to_income) 생성
pre['installment_to_income'] = pre['installment'] / (pre['annual_inc'] / 12)

# 파생변수 4. 대출규모 대비 소득(loan_to_income) 생성
pre['loan_to_income'] = pre['loan_amnt'] / pre['annual_inc']

# 파생변수 5. 회전부채 대비 소득(revol_bal_to_income) 생성
pre['revol_bal_to_income'] = pre['revol_bal'] / pre['annual_inc'] 

# 파생변수 6. 대출 당시 신용 조회 평균 (fico_mid)
pre['fico_mid'] = (pre['fico_range_low'] + pre['fico_range_high'])/2

# 파생변수 7. 신용 조회 컬럼 플래그 생성 (mths_since_last_major_derog_missing, mths_since_recent_inq)
# 결측이라면 연체가 없다는 의미
cols = [
'mths_since_last_major_derog',
'mths_since_recent_inq',
]
for col in cols:
    pre[col+"_missing"] = pre[col].isna().astype(int) # 결측값일때 1 아닐때 0, 1이라는 것은 연체가 없다는 것 


# 숫자 잘못 처리 되었을 때 NaN 처리 (분모가 0일 때 inf값 생성 가능성 있음)
new_cols = ['installment_to_income', 'loan_to_income','revol_bal_to_income']
for col in new_cols:
    pre[col] = pre[col].replace([np.inf, -np.inf], np.nan)


# 변경된 정책을 반영하여 행 제거
# 연도 자르기 
change_mask = "num_actv_bc_tl" # 연도를 자를 기준컬럼 변수 지정(활성 뱅크카드 수 컬럼)
pre = pre.dropna(subset=[change_mask])

# mo_sin_old_rev_tl_op이 null인 행 제거 
pre = pre.dropna(subset=["mo_sin_old_rev_tl_op"]) # null 1
pre = pre.dropna(subset=["num_rev_accts"]) # null 1

# 전체 계좌 평균 잔액 ($) 이 null 인행 제거 
pre = pre.dropna(subset=["avg_cur_bal"])

## 결측치 처리

In [ ]:

# 범주형 변수 처리
pre["emp_length"] = pre["emp_length"].fillna("unknown") # 근속 연수 결측치는  unknown 처리
pre["home_ownership"] = pre["home_ownership"].replace(["ANY","NONE"],"OTHER" ) # 주거 형태  OTHER 로 통합 

# 수치형 변수 처리
pre["inq_last_6mths"] = pre['inq_last_6mths'].fillna(0) #  최근 6개월간 신용 조회(hard inquiry) 횟수 결측치 1
pre["revol_util"]= pre["revol_util"].fillna(0) #리볼빙 이용률 (%) 
pre["bc_open_to_buy"]= pre["bc_open_to_buy"].fillna(0) #뱅크카드 잔여 한도 ($) 
pre["bc_util"]= pre["bc_util"].fillna(0) #뱅크카드 이용률 (%)
pre["percent_bc_gt_75"]= pre["percent_bc_gt_75"].fillna(0) #뱅크카드 75%인 뱅크카드 비율 (%)
pre["mo_sin_old_il_acct"].fillna(0)
pre["dti"] = pre["dti"].replace(999, np.nan) # dti 변경전 >> 결측치 Null 로 변경
pre.loc[pre["dti"] < 0, "dti"] = np.nan 


## 미사용 컬럼 제거

In [17]:
# 미사용 컬럼 제거

"""
제거 대상 컬럼
"""
print(f"제거 작업 전 컬럼 수 : {len(pre.columns)}개")

drop_columns = [
    # ── 그룹 1: 사후 상환 실적 ──
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'last_pymnt_amnt', 'last_pymnt_d', 'next_pymnt_d',

    # ── 그룹 2: 잔액 사후 정보 ──
    'out_prncp', 'out_prncp_inv',

    # ── 그룹 3: 상각/추심 사후 ──
    'recoveries', 'collection_recovery_fee',

    # ── 그룹 4: 사후 신용 스냅샷 ──
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',

    # ── 그룹 5: Hardship / Settlement ──
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    'orig_projected_additional_accrued_interest',
    'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term',

    # ── 그룹 6: 특별 상환 계획 ──
    'pymnt_plan', 'payment_plan_start_date',

    # ── 그룹 7: 승인/집행 결과 ──
    'funded_amnt', 'funded_amnt_inv', 'disbursement_method',

    # ── 그룹 8: 식별자 및 상수 ──
    'id', 'member_id', 'url', 'title', 'zip_code', 'policy_code',

    # ── 그룹 9: 원본 타겟 변수 ──
    'loan_status',

    # ── 그룹 10: 효과 크기 낮은 변수 ── 후보 
    #'initial_list_status', 'total_acc', 'open_acc',
    #'application_type', 'addr_state', 'earliest_cr_line', 'pub_rec',

    # ── 그룹 11: 공동 신청(Joint) 관련 ──
    'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high',
    'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc',
    'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il',
    'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths',
    'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog',

    # 2016년 이후 신규 추가된 세부 신용 리포트 항목 (13개) 
    "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m","total_bal_il"
    ,"il_util","open_rv_12m","open_rv_24m","max_bal_bc","all_util","inq_fi" 
    ,"total_cu_tl", "inq_last_12m",

    # 3차 검토 사용 불가 판정 
    "desc", "emp_title", "num_tl_120dpd_2m", 
    
    # 모델에 사용하지 않는 컬럼 제거
    'grade', 'verification_status', 'addr_state', 'delinq_2yrs',
    'inq_last_6mths', 'mths_since_last_record', 'open_acc',
    'initial_list_status', 'collections_12_mths_ex_med', 'application_type',
    'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'bc_util',
    'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_rcnt_tl', 'mort_acc',
    'num_accts_ever_120_pd', 'num_bc_sats', 'num_bc_tl', 'num_op_rev_tl',
    'num_rev_accts', 'num_sats', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
    'num_tl_op_past_12m', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
    'tax_liens', 'total_bal_ex_mort', 'total_il_high_credit_limit',
]

# 중복 검증
assert len(drop_columns) == len(set(drop_columns)), \
    f"중복 존재: {[c for c in drop_columns if drop_columns.count(c) > 1]}"

# 실제 적용
pre = pre.drop(columns=[c for c in drop_columns if c in df.columns])

print(f"총 제거 대상: {len(drop_columns)}개")
print(f"남은 컬럼: {len(pre.columns)}개")

제거 작업 전 컬럼 수 : 157개
총 제거 대상: 112개
남은 컬럼: 45개


In [ ]:
# # 전처리한 데이터 내보내기 
pre.to_csv("data/lending_club_preprocessed3.csv", index=False)